# AI工学101 — 第7回

## 欠損値・外れ値・データクリーニング 🧹

よし、第6回の**平均・分散・標準化**から続行です。

今日はモデルを作る前の、かなり実務的な仕事。

> **「現実のデータ、そんな綺麗じゃねえぞ」問題**

を処理します😂

実データには空欄、入力ミス、異常値、単位違いなどが普通に混ざります。今日はNumPyで **欠損を発見 → 状態を調査 → 除外または補完** まで自分で実装します。

**所要時間：60〜90分**
📖 講義20分 ／ 💻 実習40〜50分 ／ ✍️ 演習20分

---

## 📖 講義：`NaN` とは何者か

こんなデータが来ました。

```python
import numpy as np

X = np.array([
    [25, 150],
    [30, 180],
    [35, np.nan],
    [40, 200],
    [np.nan, 160]
])
```

`np.nan` は、

> **Not a Number**

を表します。

ここでは、

```text
年齢 | スコア
-----------
25   | 150
30   | 180
35   | ???
40   | 200
???  | 160
```

という欠損データですね。

現実にはCSVの空欄などから入ってきます。

---

# 💻 実習1：欠損値を発見する

まず、

```python
print(np.isnan(X))
```

を実行します。

結果はこんな感じ。

```text
[[False False]
 [False False]
 [False  True]
 [False False]
 [ True False]]
```

第3回でやった**Boolean Mask**そのものです。

そして、

```python
np.isnan(X).sum()
```

で欠損値の総数が分かります。

今回は、

```text
2
```

ですね。

### 列ごとにも調べられる

```python
np.isnan(X).sum(axis=0)
```

ここでは第4回の `axis` が帰ってきます。

```text
[1 1]
```

つまり、

```text
年齢   → 1個欠損
スコア → 1個欠損
```

です。

これ、もう立派な**データ品質チェック**です。

---

# 💻 実習2：`NaN` の罠

普通に平均を計算してみます。

```python
print(X.mean(axis=0))
```

すると、

```text
[nan nan]
```

になってしまいます。

なぜなら、

```text
25 + 30 + 35 + 40 + ???
```

の平均なんて計算できないから。

そこでNumPyには、

```python
np.nanmean()
```

があります。

```python
mean = np.nanmean(X, axis=0)

print(mean)
```

これは、

> **NaNを無視して平均を計算**

してくれます。

同様に、

```python
np.nanstd(X, axis=0)
np.nanmedian(X, axis=0)
```

などもあります。

---

# 💻 実習3：欠損行を削除する

欠損が含まれている行を丸ごと消してみます。

まず、

```python
missing = np.isnan(X)
```

各行に1個でも欠損があるか？

```python
bad_rows = missing.any(axis=1)

print(bad_rows)
```

イメージは、

```text
False
False
True
False
True
```

です。

では逆転。

```python
good_rows = ~bad_rows
```

`~` はBooleanを反転します。

```text
True
True
False
True
False
```

そして、

```python
X_clean = X[good_rows]

print(X_clean)
```

結果：

```text
[[ 25. 150.]
 [ 30. 180.]
 [ 40. 200.]]
```

欠損行が消えました。

---

## ここで重要な判断

「NaNなら削除！」

ではありません。

例えば100万件中10件欠損なら、削除しても大きな問題にならないかもしれません。

でも100件中40件欠損なら？

40%捨てることになります。

さらに、欠損の発生に偏りがあれば、削除によってデータ自体を歪める可能性があります。

つまり、

> **欠損処理はプログラミング問題ではなく、データについての判断でもある。**

ここはデータサイエンス色が強いところです。

---

# 💻 実習4：平均値で補完する

今度は削除せず、欠損を埋めます。

```python
X = np.array([
    [25, 150],
    [30, 180],
    [35, np.nan],
    [40, 200],
    [np.nan, 160]
], dtype=float)
```

列ごとの平均：

```python
means = np.nanmean(X, axis=0)

print(means)
```

次に、

```python
X_filled = X.copy()
```

コピーを作っておきます。

そして欠損位置を取得。

```python
rows, cols = np.where(np.isnan(X_filled))
```

確認：

```python
print(rows)
print(cols)
```

それぞれの欠損について、

```python
X_filled[rows, cols] = means[cols]
```

とすると……

```python
print(X_filled)
```

欠損が平均値で埋まります。

最後に、

```python
print(np.isnan(X_filled).sum())
```

結果が、

```text
0
```

なら成功です。

---

# 📖 外れ値は欠損より難しい

次はこちら。

```python
scores = np.array([
    48, 52, 49, 51, 50, 53, 47, 500
])
```

500。

君はたぶん、

> 「おい。」

と思うでしょう。

人間には一瞬で怪しく見える。

しかしコンピュータは、

```text
500も数字ですが？
```

という顔をしています。

欠損値なら `NaN` という印がありますが、**外れ値には「私は外れ値です」という印がありません。**

ここが難しい。

---

# 💻 実習5：標準偏差を使って調べる

第6回の知識を使います。

```python
mean = scores.mean()
std = scores.std()

z = (scores - mean) / std

print(z)
```

これは各値が、

> 平均から標準偏差何個分離れているか

を表しています。

一般に大きな絶対値のZスコアは「怪しい値」を探す手掛かりになります。

例えば、

```python
np.abs(z)
```

で符号を消せます。

```python
mask = np.abs(z) > 2

print(scores[mask])
```

として候補を抽出できます。

ただし、

> `|z| > 2` なら必ず異常

という意味ではありません。

統計的に珍しい値と、データとして間違っている値は別物です。

身長250cmなら非常に珍しいですが、存在不能とは限らない。

身長2500cmなら単位ミスを疑う。

**統計だけでは意味までは判断できない**んですね。

---

# 💻 実習6：中央値という武器

さっきのデータで、

```python
print(scores.mean())
print(np.median(scores))
```

を比較してください。

500によって平均は大きく引っ張られます。

中央値はかなり頑丈です。

これを、

> **外れ値に対してロバスト**

と表現します。

今後、

```text
mean
median
```

を見たとき、

「どっちが正しい？」

ではなく、

> **このデータにはどちらが適切？**

と考えるのがデータサイエンスの姿勢です。

---

# 💻 実習7：小さなクリーニングパイプライン

全部つなげます。

```python
X = np.array([
    [20, 55],
    [25, 60],
    [30, np.nan],
    [35, 72],
    [40, 75],
    [45, 500],
    [50, 82]
], dtype=float)
```

まず欠損数。

```python
print(np.isnan(X).sum(axis=0))
```

平均値補完：

```python
means = np.nanmean(X, axis=0)

X_clean = X.copy()

rows, cols = np.where(np.isnan(X_clean))

X_clean[rows, cols] = means[cols]
```

確認：

```python
print(X_clean)
```

そしてスコア列だけ取り出す。

```python
scores = X_clean[:, 1]
```

中央値などを調査。

```python
print("mean:", scores.mean())
print("median:", np.median(scores))
print("std:", scores.std())
```

これで、

```text
生データ
 ↓
欠損調査
 ↓
欠損補完
 ↓
外れ値調査
 ↓
モデル投入可能なデータ
```

という流れを一通り経験しました。

---

# ✍️ 演習：汚れた研究データを救出せよ

今日のデータはこちら。

```python
X = np.array([
    [22, 120],
    [25, np.nan],
    [28, 135],
    [np.nan, 142],
    [34, 138],
    [37, 900],
    [40, 150]
], dtype=float)
```

各行は、

```text
[年齢, 測定値]
```

です。

今回は6問。

1. `np.isnan()` を使って欠損値の位置を確認してください。
2. 列ごとの欠損数を求めてください。
3. `np.nanmean()` で列ごとの平均を求めてください。
4. 欠損値をその列の平均値で補完してください。
5. 測定値列について `mean` と `median` を比較してください。
6. `900` を外れ値候補として除外した場合、平均がどう変化するか確認してください。

最後の問では、

```python
scores < 900
```

のようなBoolean Maskを使って構いません。

ただし、**「900だから削除」まで自動化する必要はありません。**

今回は「怪しいので調査対象から一時除外する」という扱いです。

---

## 🌿 第7回の核心

ここまで来ると、NumPyの個々の機能がかなり接続されてきました。

```text
Boolean Mask ─┐
axis ─────────┤
平均・標準偏差 ┤
ブロードキャスト ┤
               ↓
       データクリーニング
               ↓
          機械学習
```

そして今日いちばん重要なのは、コードよりむしろ、

> **データ前処理には「計算」と「判断」の両方がある**

ということです。

`np.isnan()` は欠損を発見してくれる。でも、その行を捨てるべきかは教えてくれない。

Zスコアは珍しい値を発見してくれる。でも、それが測定ミスなのか貴重な観測なのかは教えてくれない。

この境界が、**単なるPython操作からデータサイエンスへ移るところ**です。

次の**第8回は「乱数・サンプリング・再現性」**。`np.random`、seed、シャッフル、訓練/評価データという考え方に入り、いよいよNumPy編の後半から scikit-learn への橋を架け始めます。

リュール先生、着々と機械学習の玄関まで湿った肉を運んでおります👨‍🏫🥝

In [ ]:
# # ✍️ 演習：汚れた研究データを救出せよ

# 今日のデータはこちら。

# ```python
# X = np.array([
#     [22, 120],
#     [25, np.nan],
#     [28, 135],
#     [np.nan, 142],
#     [34, 138],
#     [37, 900],
#     [40, 150]
# ], dtype=float)
# ```

# 各行は、

# ```text
# [年齢, 測定値]
# ```

# です。

# 今回は6問。

# 1. `np.isnan()` を使って欠損値の位置を確認してください。
# 2. 列ごとの欠損数を求めてください。
# 3. `np.nanmean()` で列ごとの平均を求めてください。
# 4. 欠損値をその列の平均値で補完してください。
# 5. 測定値列について `mean` と `median` を比較してください。
# 6. `900` を外れ値候補として除外した場合、平均がどう変化するか確認してください。

# 最後の問では、

# ```python
# scores < 900
# ```

# のようなBoolean Maskを使って構いません。
# ただし、**「900だから削除」まで自動化する必要はありません。**
# 今回は「怪しいので調査対象から一時除外する」という扱いです。